# Greenhouse Temperature Analyser
Parses the multi-sensor CSV exported from the climate system (AV, RV, VD, kastemperatuur).

In [1]:
import pandas as pd

CSV_FILE = 'greenhouse_temp_2026-05-28_to_12-18-07.csv'

# Read raw CSV (semicolon-separated, 4 header rows to skip)
raw = pd.read_csv(CSV_FILE, sep=';', header=None, skiprows=4)

# The file interleaves 4 sensor groups: each group occupies 2 cols + 1 empty col
# Layout: [ts0, val0, empty, ts1, val1, empty, ts2, val2, empty, ts3, val3, empty]
sensor_names = ['AV', 'RV', 'VD', 'kastemperatuur']
offsets = [0, 3, 6, 9]  # starting column index for each group

frames = []
for name, col in zip(sensor_names, offsets):
    chunk = raw.iloc[:, col:col+2].copy()
    chunk.columns = ['timestamp', name]
    chunk['timestamp'] = pd.to_datetime(chunk['timestamp'], utc=True)
    chunk[name] = pd.to_numeric(
        chunk[name].astype(str).str.replace(',', '.'), errors='coerce'
    )
    frames.append(chunk.set_index('timestamp'))

df = pd.concat(frames, axis=1).sort_index()
df.index = df.index.tz_convert('Europe/Amsterdam')
print(f'Rows: {len(df)}  |  Date range: {df.index[0].date()} to {df.index[-1].date()}')
df.head(10)

Rows: 1308  |  Date range: 2026-04-04 to 2026-05-28


,AV,RV,VD,kastemperatuur
timestamp,,,,
2026-04-04 00:00:00+02:00,12.133333,94.000000,0.775000,15.041667
2026-04-04 01:00:00+02:00,12.150000,94.250000,0.708333,14.966667
2026-04-04 02:00:00+02:00,12.341667,93.000000,0.908333,15.416667
2026-04-04 03:00:00+02:00,12.058333,91.083333,1.175000,15.433333
2026-04-04 04:00:00+02:00,11.833333,89.500000,1.391667,15.400000
2026-04-04 05:00:00+02:00,11.850000,86.500000,1.825000,15.975000
2026-04-04 06:00:00+02:00,12.016667,86.250000,1.900000,16.233333
2026-04-04 07:00:00+02:00,12.600000,85.333333,2.141667,17.175000
2026-04-04 08:00:00+02:00,13.166667,85.500000,2.191667,17.875000


In [2]:
df.describe()

,AV,RV,VD,kastemperatuur
count,1308.000000,1308.000000,1308.000000,1308.000000
mean,13.277472,76.807212,4.491278,19.788500
std,2.512643,11.444402,3.334731,4.307656
min,5.633333,23.083333,0.708333,13.566667
25%,11.258333,71.562500,2.089583,15.741667
50%,12.679167,81.000000,3.100000,18.929167
75%,14.966667,84.604167,6.077083,23.968750
max,22.441667,94.250000,18.591667,31.666667


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)

labels = {
    'kastemperatuur': ('Greenhouse temperature (C)', 'tab:red'),
    'RV':             ('Relative humidity (%)',      'tab:blue'),
    'AV':             ('Ventilation (AV)',            'tab:green'),
    'VD':             ('Vapour deficit (VD)',         'tab:orange'),
}

for ax, (col, (ylabel, color)) in zip(axes, labels.items()):
    ax.plot(df.index, df[col], color=color, linewidth=0.8)
    ax.set_ylabel(ylabel, fontsize=9)
    ax.grid(True, alpha=0.3)

axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%d %b'))
fig.autofmt_xdate()
fig.suptitle('Greenhouse climate - hourly data', fontsize=13)
plt.tight_layout()
plt.show()